# 02. Обучение модели рекомендаций

Теперь попробуем предсказать следующую покупку клиента. Модель должна расставить 24 продукта так, чтобы нужные оказались как можно выше в списке.

Главная метрика — MAP@7. Дополнительно посмотрим Recall@7, Precision@7, покрытие каталога и macro PR-AUC, чтобы не делать вывод по одному показателю.

Ноутбук использует весь исходный CSV. Здесь можно обучить несколько вариантов, сравнить их и сохранить лучшую модель вместе с отчётами. Результаты ячеек очищены, поэтому для просмотра чисел ноутбук нужно запустить заново.

In [3]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import hashlib
import json
import os
import random
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate_root = PROJECT_ROOT.parent
    if (candidate_root / "src").is_dir():
        PROJECT_ROOT = candidate_root
    else:
        raise RuntimeError("Запустите ноутбук из корня проекта или каталога notebooks")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from bank_recommender.constants import (
    DATE_COLUMN,
    ID_COLUMN,
    PRODUCT_COLUMNS,
    PRODUCT_DESCRIPTIONS_RU,
)
from bank_recommender.data import build_temporal_pairs, read_snapshots
from bank_recommender.features import engineer_features
from bank_recommender.metrics import evaluate_ranking
from bank_recommender.model import (
    BankProductRecommender,
    PopularityRecommender,
    ranked_recommendations,
)

SEED = 42
TOP_K = 7
VALIDATION_PERIOD = "2016-04"
random.seed(SEED)
np.random.seed(SEED)

raw_data_path = Path(os.getenv("DATA_PATH", "train_ver2.csv")).expanduser()
DATA_PATH = (
    raw_data_path if raw_data_path.is_absolute() else PROJECT_ROOT / raw_data_path
)
MODEL_MAX_ITER = int(os.getenv("MODEL_MAX_ITER", "12"))
FINAL_MODEL_MAX_ITER = int(os.getenv("FINAL_MODEL_MAX_ITER", "60"))
if MODEL_MAX_ITER < 1 or FINAL_MODEL_MAX_ITER < 1:
    raise ValueError("Число итераций должно быть положительным")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Датасет не найден: {DATA_PATH}")

MODEL_CONFIGS = {
    "sgd_alpha_1e-4": {"alpha": 0.0001},
    "sgd_alpha_1e-3": {"alpha": 0.001},
}
BLEND_WEIGHTS = [0.0, 0.25, 0.5, 0.75]
raw_artifact_path = Path(
    os.getenv("MODEL_ARTIFACT_PATH", "ml_models/model.joblib")
).expanduser()
ARTIFACT_PATH = (
    raw_artifact_path
    if raw_artifact_path.is_absolute()
    else PROJECT_ROOT / raw_artifact_path
)
raw_report_dir = Path(os.getenv("MODEL_REPORT_DIR", "reports")).expanduser()
REPORT_DIR = (
    raw_report_dir if raw_report_dir.is_absolute() else PROJECT_ROOT / raw_report_dir
)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv(
    "MLFLOW_EXPERIMENT_NAME",
    "bank-product-recommendation",
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

print(f"Данные: {DATA_PATH}")
print(
    "Режим загрузки: все клиенты; "
    f"seed={SEED}; сравнение max_iter={MODEL_MAX_ITER}; "
    f"финал max_iter={FINAL_MODEL_MAX_ITER}; "
    f"holdout={VALIDATION_PERIOD}->2016-05"
)

Данные: C:\Users\nike1\projects\mle-pr-final\train_ver2.csv
Режим загрузки: все клиенты; seed=42; сравнение max_iter=12; финал max_iter=60; holdout=2016-04->2016-05


## 1. Как определяем новую покупку

Новая покупка для клиента $u$ и продукта $p$ считается так:

$$y_{u,p,t+1}=\max(product_{u,p,t+1}-product_{u,p,t}, 0).$$

Покупкой считаем только переход `0→1`. Если продукт остался, его по-прежнему нет или клиент от него отказался, цель равна нулю. Сравниваем только соседние месяцы, где известны все 24 продуктовых флага.

Модель видит профиль и продукты клиента только за месяц $t$. Данные следующего месяца в признаки не попадают. Обучаемся на ранних месяцах, а по апрелю 2016 года предсказываем покупки мая. Все правила обработки настраиваем только на обучающих данных.

In [4]:
snapshots = read_snapshots(DATA_PATH)
pairs = build_temporal_pairs(snapshots)
x_train, y_train, x_valid, y_valid = pairs.split(VALIDATION_PERIOD)

valid_periods = x_valid[DATE_COLUMN].dt.to_period("M")
if not valid_periods.eq(pd.Period(VALIDATION_PERIOD, freq="M")).all():
    raise AssertionError("Holdout содержит строки вне апреля 2016 года")

split_summary = pd.DataFrame(
    {
        "train": [
            len(x_train),
            x_train[ID_COLUMN].nunique(),
            str(x_train[DATE_COLUMN].min().to_period("M")),
            str(x_train[DATE_COLUMN].max().to_period("M")),
            int(y_train[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_train[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
        "validation (апрель→май)": [
            len(x_valid),
            x_valid[ID_COLUMN].nunique(),
            str(x_valid[DATE_COLUMN].min().to_period("M")),
            str(x_valid[DATE_COLUMN].max().to_period("M")),
            int(y_valid[PRODUCT_COLUMNS].to_numpy().sum()),
            float((y_valid[PRODUCT_COLUMNS].sum(axis=1) > 0).mean()),
        ],
    },
    index=[
        "пар",
        "клиентов",
        "минимальный feature-месяц",
        "максимальный feature-месяц",
        "событий 0→1",
        "доля клиентов с покупкой",
    ],
)
display(split_summary)

,train,validation (апрель→май)
пар,11742921,926663
клиентов,938929,926663
минимальный feature-месяц,2015-01,2016-04
максимальный feature-месяц,2016-03,2016-04
событий 0→1,525805,35843
доля клиентов с покупкой,0.03565,0.03008


## 2. Проверяем, насколько редки покупки

Посчитаем новые подключения каждого продукта в обучающей и проверочной частях. Так сразу станет видно, для каких продуктов почти нет примеров.

Если в каком-то периоде покупок продукта не было, не удаляем его из каталога. Это значит только то, что за этот месяц новых подключений не встретилось.

In [5]:
target_prevalence = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "train_events": [int(y_train[p].sum()) for p in PRODUCT_COLUMNS],
        "train_rate": [float(y_train[p].mean()) for p in PRODUCT_COLUMNS],
        "validation_events": [int(y_valid[p].sum()) for p in PRODUCT_COLUMNS],
        "validation_rate": [float(y_valid[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("train_events", ascending=False, ignore_index=True)
display(target_prevalence.head(10))
display(target_prevalence.tail(5))

,product,description,train_events,train_rate,validation_events,validation_rate
0,ind_recibo_ult1,Счёт прямого дебета,142958,0.01217,10163,0.01097
1,ind_nom_pens_ult1,Пенсионные обязательства,79196,0.00674,5513,0.00595
2,ind_nomina_ult1,Зарплатный счёт,68265,0.00581,5488,0.00592
3,ind_cco_fin_ult1,Текущий счёт,65223,0.00555,3854,0.00416
4,ind_tjcr_fin_ult1,Кредитная карта,64870,0.00552,4248,0.00458
5,ind_cno_fin_ult1,Зарплатный проект,34752,0.00296,2346,0.00253
6,ind_ecue_fin_ult1,Цифровой счёт,23652,0.00201,2709,0.00292
7,ind_dela_fin_ult1,Долгосрочный депозит,12612,0.00107,46,0.00005
8,ind_reca_fin_ult1,Налоговый счёт,8950,0.00076,279,0.00030
9,ind_ctma_fin_ult1,Особый счёт 3,6342,0.00054,512,0.00055


,product,description,train_events,train_rate,validation_events,validation_rate
19,ind_cder_fin_ult1,Деривативный счёт,131,0.00001,5,0.00001
20,ind_hip_fin_ult1,Ипотека,72,0.00001,3,0.00000
21,ind_viv_fin_ult1,Счёт домохозяйства,63,0.00001,7,0.00001
22,ind_aval_fin_ult1,Банковская гарантия,4,0.00000,0,0.00000
23,ind_ahor_fin_ult1,Сберегательный счёт,1,0.00000,1,0.00000


## 3. Начинаем с простого ориентира

Сначала возьмём простой baseline — популярность новых покупок. Он считает переходы `0→1` в обучающих данных и предлагает всем клиентам одинаковый порядок продуктов.

Такой подход не учитывает особенности клиента, но даёт понятную точку отсчёта. Сложная модель полезна только тогда, когда обгоняет baseline на следующем месяце. Продукты, которые уже есть у клиента, в рекомендации не попадают.

In [6]:
baseline_started = perf_counter()
baseline = PopularityRecommender().fit(x_train, y_train)
baseline_fit_seconds = perf_counter() - baseline_started
baseline_scores = baseline.predict_scores(x_valid)
comparison = {
    "popularity": evaluate_ranking(y_valid, baseline_scores, owned=x_valid, k=TOP_K)
}
comparison["popularity"]["fit_seconds"] = baseline_fit_seconds
display(pd.DataFrame(comparison).T)

,map_at_7,precision_at_7,recall_at_7,catalog_coverage_at_7,customers_with_purchase_rate,macro_pr_auc,fit_seconds
popularity,0.01935,0.00534,0.02892,0.83333,0.03008,0.00184,0.82178


## 4. Обучаем несколько моделей

Сравним два варианта SGD с разной силой регуляризации. В каждом варианте для каждого продукта обучается отдельный `SGDClassifier`. Признаки готовятся одинаково:

- добавим месяц, год и характеристики текущего портфеля;
- числовые пропуски заполним медианами и отмасштабируем значения;
- для пропусков в категориях создадим отдельное значение, а сами категории закодируем;
- новые категории при прогнозе не будут вызывать ошибку.

Для сравнения используем небольшое число итераций, чтобы не ждать слишком долго. После выбора лучший вариант обучим ещё раз с финальным числом итераций.

In [ ]:
display(pd.DataFrame(MODEL_CONFIGS).T.rename_axis("model"))
model_candidates = {}
model_scores = {}
model_fit_seconds = {}

for model_name, config in MODEL_CONFIGS.items():
    model = BankProductRecommender(
        alpha=config["alpha"],
        max_iter=MODEL_MAX_ITER,
        prior_weight=0.0,
        random_seed=SEED,
    )
    model_started = perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        warnings.filterwarnings("ignore", message="Label .*", category=UserWarning)
        model.fit(x_train, y_train)
    model_fit_seconds[model_name] = perf_counter() - model_started

    scores = model.predict_scores(x_valid)
    expected_shape = (len(x_valid), len(PRODUCT_COLUMNS))
    if scores.shape != expected_shape:
        raise AssertionError("Модель вернула матрицу score неожиданного размера")
    model_candidates[model_name] = model
    model_scores[model_name] = scores
    print(
        f"Обучена {model_name}: alpha={config['alpha']}; "
        f"время={model_fit_seconds[model_name]:.1f} с; score={scores.shape}"
    )

,alpha
model,
sgd_alpha_1e-4,0.00010
sgd_alpha_1e-3,0.00100


Обучена sgd_alpha_1e-4: alpha=0.0001; время=1220.7 с; score=(926663, 24)


In [ ]:
reference_candidate = next(iter(model_candidates.values()))
pipeline_steps = pd.DataFrame(
    [
        {"этап": name, "класс": type(step).__name__}
        for name, step in reference_candidate.pipeline_.steps
    ]
)
preprocessor = reference_candidate.pipeline_.named_steps["preprocessing"]
preprocessing_steps = pd.DataFrame(
    [
        {
            "ветка": name,
            "pipeline": " → ".join(step_name for step_name, _ in transformer.steps),
            "число входных признаков": len(columns),
        }
        for name, transformer, columns in preprocessor.transformers_
        if hasattr(transformer, "steps")
    ]
)
engineered_preview = engineer_features(x_train.head(3))
display(pipeline_steps)
display(preprocessing_steps)
print(
    f"После feature engineering: {engineered_preview.shape[1]} признаков; "
    f"классификаторов/меток: {len(PRODUCT_COLUMNS)}"
)

## 5. Сравниваем несколько вариантов

Теперь сравним baseline, оба варианта SGD и их смеси с общей популярностью. Смешивание иногда помогает редким продуктам:



Лучший вариант SGD выберем по MAP@7 на переходе апрель→май, а popularity оставим как точку отсчёта. Recall@7 покажет, сколько покупок мы нашли, Precision@7 — сколько рекомендаций оказались полезными, покрытие — насколько разнообразен список, а macro PR-AUC — как модель работает с отдельными продуктами. Рядом выведем время обучения каждой модели.

In [ ]:
experiment_details = {"popularity": {"model_name": "popularity", "prior_weight": 1.0}}
for model_name, scores in model_scores.items():
    for weight in BLEND_WEIGHTS:
        experiment_name = f"{model_name}_prior_{weight:.2f}"
        blended_scores = (1.0 - weight) * scores + weight * baseline_scores
        comparison[experiment_name] = evaluate_ranking(
            y_valid, blended_scores, owned=x_valid, k=TOP_K
        )
        comparison[experiment_name]["fit_seconds"] = model_fit_seconds[model_name]
        experiment_details[experiment_name] = {
            "model_name": model_name,
            "prior_weight": weight,
        }

metric_columns = [
    "map_at_7",
    "recall_at_7",
    "precision_at_7",
    "catalog_coverage_at_7",
    "macro_pr_auc",
    "customers_with_purchase_rate",
    "fit_seconds",
]
comparison_frame = (
    pd.DataFrame(comparison).T[metric_columns].sort_values("map_at_7", ascending=False)
)
learned_names = [name for name in comparison if name != "popularity"]
best_experiment = max(
    learned_names,
    key=lambda name: comparison[name]["map_at_7"],
)
best_model_name = experiment_details[best_experiment]["model_name"]
best_blend_weight = experiment_details[best_experiment]["prior_weight"]
best_model_scores = (1.0 - best_blend_weight) * model_scores[
    best_model_name
] + best_blend_weight * baseline_scores

display(comparison_frame)
print(f"Лучший обучаемый вариант по MAP@7: {best_experiment}")

In [ ]:
plot_metrics = (
    comparison_frame[["map_at_7", "recall_at_7", "precision_at_7"]]
    .reset_index(names="experiment")
    .melt(id_vars="experiment", var_name="metric", value_name="value")
)
plt.figure(figsize=(14, 6))
sns.barplot(data=plot_metrics, x="experiment", y="value", hue="metric")
plt.title("Сравнение моделей на holdout апрель→май")
plt.xlabel("Эксперимент")
plt.ylabel("Значение метрики")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Метрика")
plt.tight_layout()
plt.show()

## 6. Проверяем рекомендации

Модель не должна советовать продукт, который уже есть у клиента. Проверим это на всей проверочной части и посмотрим один пример рекомендации.

Заодно убедимся, что в списке нет повторов, а новая покупка `0→1` не пересекается с текущим портфелем.

In [ ]:
recommendations = ranked_recommendations(best_model_scores, x_valid, k=TOP_K)
owned_matrix = x_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
target_matrix = y_valid[PRODUCT_COLUMNS].to_numpy(dtype=bool)
if np.logical_and(owned_matrix, target_matrix).any():
    raise AssertionError("Target 0→1 пересекается с текущими владениями")

violations = []
for row_number, row_recommendations in enumerate(recommendations):
    owned_products = {
        product
        for product in PRODUCT_COLUMNS
        if owned_matrix[row_number, PRODUCT_COLUMNS.index(product)]
    }
    recommended_products = [item["product"] for item in row_recommendations]
    overlap = owned_products.intersection(recommended_products)
    if overlap or len(recommended_products) != len(set(recommended_products)):
        violations.append((row_number, sorted(overlap)))

if violations:
    raise AssertionError(f"Найдены нарушения фильтрации: {violations[:3]}")

example_row = 0
example_owned = [
    PRODUCT_DESCRIPTIONS_RU[p]
    for p in PRODUCT_COLUMNS
    if bool(x_valid.iloc[example_row][p])
]
example_recommended = [
    PRODUCT_DESCRIPTIONS_RU[item["product"]] for item in recommendations[example_row]
]
example = pd.Series(
    {
        "ncodpers": int(x_valid.iloc[example_row][ID_COLUMN]),
        "уже есть": example_owned,
        "top-7 рекомендаций": example_recommended,
    },
)
print(f"Проверено строк holdout: {len(recommendations)}; нарушений: 0")
display(example.to_frame(name="значение"))

## 7. Переобучаем и сохраняем лучшую модель

Теперь объединим обучающую и проверочную части и ещё раз обучим лучший вариант SGD. После запуска следующей ячейки модель сохранится в `ml_models/model.joblib`, а таблица сравнения и карточка модели — в `reports`.

Пути можно изменить через `MODEL_ARTIFACT_PATH` и `MODEL_REPORT_DIR`. Если задан `MLFLOW_TRACKING_URI`, этот же запуск будет записан в MLflow.

In [ ]:
all_features = pd.concat([x_train, x_valid], ignore_index=True)
all_targets = pd.concat([y_train, y_valid], ignore_index=True)
best_config = MODEL_CONFIGS[best_model_name]
final_model = BankProductRecommender(
    alpha=best_config["alpha"],
    max_iter=FINAL_MODEL_MAX_ITER,
    prior_weight=best_blend_weight,
    random_seed=SEED,
    model_version="sgd-ovr-v1",
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", message="Label .*", category=UserWarning)
    final_model.fit(all_features, all_targets)

pair_periods = pairs.periods.astype(str)
metadata = {
    "model_version": final_model.model_version,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "rank products newly acquired in the next calendar month",
    "target_definition": "max(product[t+1] - product[t], 0)",
    "pair_quality_rule": "all_product_flags_known_in_both_months",
    "validation_period": VALIDATION_PERIOD,
    "training_period_min": pair_periods.min(),
    "training_period_max": pair_periods.max(),
    "forecast_snapshot": str(snapshots[DATE_COLUMN].max().date()),
    "forecast_month": str(snapshots[DATE_COLUMN].max().to_period("M") + 1),
    "data_scope": "full_dataset",
    "loading_mode": "single_read",
    "random_seed": SEED,
    "snapshot_rows": len(snapshots),
    "training_pairs": len(x_train),
    "validation_pairs": len(x_valid),
    "refit_pairs": len(all_features),
    "candidate_models": MODEL_CONFIGS,
    "selected_experiment": best_experiment,
    "selected_model": best_model_name,
    "selected_alpha": best_config["alpha"],
    "selected_prior_weight": best_blend_weight,
    "comparison_max_iter": MODEL_MAX_ITER,
    "final_max_iter": FINAL_MODEL_MAX_ITER,
    "primary_metric": f"map_at_{TOP_K}",
    "product_columns": PRODUCT_COLUMNS,
    "validation_metrics": comparison[best_experiment],
    "baseline_metrics": comparison["popularity"],
    "new_product_events_train": {
        product: int(y_train[product].sum()) for product in PRODUCT_COLUMNS
    },
}
final_model.training_metadata = metadata

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
temporary_artifact = ARTIFACT_PATH.with_suffix(ARTIFACT_PATH.suffix + ".tmp")
joblib.dump(final_model, temporary_artifact, compress=3)
temporary_artifact.replace(ARTIFACT_PATH)
metadata["artifact_sha256"] = hashlib.sha256(ARTIFACT_PATH.read_bytes()).hexdigest()

comparison_path = REPORT_DIR / "model_comparison.json"
metadata_path = REPORT_DIR / "model_metadata.json"
summary_path = REPORT_DIR / "training_summary.json"
comparison_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

mlflow_run_id = None
mlflow_error = None
if MLFLOW_TRACKING_URI:
    try:
        import mlflow
        import mlflow.sklearn
        from mlflow.models import infer_signature

        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
        input_example = all_features.head(5).copy()
        signature = infer_signature(
            input_example,
            final_model.predict_scores(input_example),
        )
        with mlflow.start_run(run_name=f"notebook-{best_experiment}") as run:
            mlflow.log_params(
                {
                    "selected_experiment": best_experiment,
                    "alpha": best_config["alpha"],
                    "prior_weight": best_blend_weight,
                    "max_iter": FINAL_MODEL_MAX_ITER,
                    "validation_period": VALIDATION_PERIOD,
                    "random_seed": SEED,
                    "data_scope": "full_dataset",
                }
            )
            for experiment_name, metrics in comparison.items():
                for metric_name, value in metrics.items():
                    mlflow.log_metric(
                        f"{experiment_name}.{metric_name}",
                        float(value),
                    )
            mlflow.log_artifact(str(comparison_path), artifact_path="reports")
            mlflow.log_artifact(str(metadata_path), artifact_path="reports")
            mlflow.sklearn.log_model(
                sk_model=final_model,
                artifact_path="model",
                signature=signature,
                input_example=input_example,
                code_paths=[str(PROJECT_ROOT / "src")],
            )
            mlflow_run_id = run.info.run_id
    except Exception as error:
        mlflow_error = f"{type(error).__name__}: {error}"

training_summary = {
    "artifact_path": str(ARTIFACT_PATH),
    "metadata_path": str(metadata_path),
    "comparison_path": str(comparison_path),
    "selected_experiment": best_experiment,
    "validation_metrics": comparison[best_experiment],
    "baseline_metrics": comparison["popularity"],
    "mlflow_run_id": mlflow_run_id,
    "mlflow_error": mlflow_error,
}
summary_path.write_text(
    json.dumps(training_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

display(comparison_frame)
display(pd.Series(training_summary, name="значение").to_frame())
print(f"Модель сохранена: {ARTIFACT_PATH}")
print(f"Отчёты сохранены: {REPORT_DIR}")

## 9. Что получилось

- Baseline и два варианта SGD сравниваются на одном периоде: по апрелю предсказываем покупки мая.
- Лучший вариант SGD выбирается по MAP@7, а popularity остаётся простой точкой отсчёта. Остальные метрики помогают заметить проблемы с редкими продуктами и разнообразием рекомендаций.
- Все шаги подготовки признаков собраны вместе и настраиваются только на обучающих данных.
- Уже подключённые продукты и повторы не попадают в рекомендации.
- Для повторяемости результата используется `seed=42`.
- Лучшая модель, таблица сравнения и отчёты сохраняются прямо из ноутбука.